# Leidenfrost Testcase
This is a testcase for evaporation. It is derived from the "film boiling" testcase.  
There a thin vapor film is situated above a heated wall.  
The flat liquid-vapor interface is disturbed according to the largest rayleigh-taylor instability wavelength.  
As the liquid evaporated a vapor bubble is forming and eventually emerging from the film.  
references for this testcase (in some points they possess some discrepancies)  
`10.1016/j.jcp.2006.07.035` (G)  
`10.1006/jcpn.200.6481` (WW)     
Here instead of a vapor film a liquid droplet is positioned above the heated wall.  

Note the following:
* We take the case with $\Delta T = 5K$
* We use the value for $h_v$ given in (WW)

In [1]:
//#r "..\..\binaries\BoSSSpad.dll"
//#r "BoSSSpadMiaoHeat\BoSSSpad.dll"
#r "d:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeCFL\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
//using BoSSS.Application.XNSFE_Solver;
//using ZwoLevelSetSolver;
//using ZwoLevelSetSolver.SolidPhase;
using HFSISolver;
using HFSISolver.SolidPhase;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
string proj = "ZLS-Leidenfrost-3D-Paper-Soft";

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @D:\Users\miao\Documents\Database\Deployment
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC3, @\\dc3\userspace\miao\cluster\Deployment"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC3, @\\dc3\userspace\miao\cluster\Deployment"
3,BoSSS.Application.BoSSSpad.SlurmClient,"SlurmClient fdycluster : miao@fdycluster, Slurm account: NONE"
4,BoSSS.Application.BoSSSpad.SlurmClient,"SlurmClient fdycluster-Temp : miao@fdycluster, Slurm account: NONE"


In [4]:
static var myBatch = ExecutionQueues[3];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("ZLS-Leidenfrost-3D-Paper-Soft");

In [5]:
BoSSSshell.WorkflowMgm.Init("ZLS-Leidenfrost-3D-Paper-Soft");

Project name is set to 'ZLS-Leidenfrost-3D-Paper-Soft'.
Default Execution queue is chosen for the database.


Opening existing database '\\dc3\userspace\miao\cluster\ZLS-Leidenfrost-3D-Paper-Soft'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database '\\fdycluster\miao\databases\ZLS-Leidenfrost-3D-Paper-Soft'.


In [7]:
//string proj = "XNSFE-Leidenfrost-Test-1";
//BoSSSshell.WorkflowMgm.Init(proj);
//BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [8]:
//var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
//sessions

## Clean up old simulations

In [9]:
// #r "..\binaries\ValidationTestRunner.dll"

In [10]:
// ValidationTestRunner.ValidationTestRunnerMain.DeleteDatabaseAndDeploymentsWhenOld(
//                "XNSFE-FilmBoiling",
//                "XNSFE-FilmBoiling*",
//                "delete_XNSFE-FilmBoiling", new TimeSpan(days: 25, hours: 1, minutes: 0, seconds: 1));

## Grid and Boundary/Initial Value configuration

In [11]:
static class Constants {
    // careful order of declaration matters!!!
    // material parameters
    public static double rho_A = 200.0;
    public static double rho_B = 5.0;

    public static double mu_A = 0.3;
    public static double mu_B = 0.005;

    public static double Sigma = 0.1;

    public static double c_A = 400.0;
    public static double c_B = 200.0;

    public static double k_A = 40.0;
    public static double k_B = 1.0;

    public static double hVap = 10000;
    public static double T_sat = 0; // 500K, shifted
    public static double T_wall = T_sat + 5.0;

    //public static double alpha_A = k_A / (c_A * rho_A);
    //public static double alpha_B = k_B / (c_B * rho_B);
    //public static double eps = 1.0 - rho_B / rho_A;

    public static double g = 9.81;

    public static double lambda = 2 * Math.PI * Math.Sqrt((3 * Sigma) / (g * Math.Abs(rho_A - rho_B)));
    public static double L = lambda / 2.0;

    public static double ST = 0.0195;//0.0294;//L * 2.0 / 3.0;//0.03; // 0.01; Solid Thickness
    public static double X0 = 0 * L;
    public static double R = L / 3.0;//L /6.0 / 4.0 / 3.0;
    public static double Y0 = R * 1.25 + ST;

    // capillary timestep, for finest res + highest degree, use this, for comparability?! 
    // Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) =>
        0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [12]:
Constants.L

0.03934220442504576

In [13]:
Constants.R

0.013114068141681921

In [14]:
Constants.ST

0.0195

In [15]:
Math.Sqrt((Constants.Sigma / (Constants.rho_A * Constants.g)))

0.007139215614635323

In [16]:
0.0117 / Math.Sqrt((Constants.Sigma / (Constants.rho_A * Constants.g))) //Dimensionlessly maximum radius https://doi.org/10.1017/jfm.2022.66

1.6388355012019968

In [17]:
Constants.rho_A * 1.5 *1.5 * Constants.R / Constants.Sigma

59.01330663756864

In [18]:
Constants.k_B * Constants.mu_B * 5.0 / (Constants.Sigma * Constants.rho_B * Math.Sqrt((Constants.Sigma / (Constants.rho_A * Constants.g))) * Constants.hVap)//The evaporation number

0.0007003570517957251

In [19]:
(Constants.k_B*Constants.mu_B*5)/(Constants.Sigma*Constants.hVap*Constants.rho_B*Math.Sqrt((Constants.Sigma / (Constants.rho_A * Constants.g)))) //The (dimensionless) evaporation number

0.0007003570517957251

In (WW) $h_v = 10^4$ in (G) $h_v = 10^3$ which is right?  
Form the plots in (G) we approximate  
$\Delta V(t = 0.455) \approx \Pi * 0.02^2 \approx 1.257*10^{-3}$  
Further:  
$\Delta V(t = 0.455) \approx \frac{\Delta m}{rho_v}$  
$\Delta m(t) \approx \frac{k_v \frac{\Delta T}{\delta}}{h_v}l_{\mathfrak{J}} t \approx 6.283*10^{-3}$
$h_v \approx \frac{k_v \frac{\Delta T}{\delta}}{\Delta m}l_{\mathfrak{J}} 0.455 \approx 11500$  
where:  
$l_{\mathfrak{J}} \approx \lambda$  
$\delta \approx 4/128 \lambda$  
We conclude, that $h_v = 10^4$ has to be used!  

In [20]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 *res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 6 * Constants.L, 6*res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergence(int res, IDatabaseInfo myDb){
        //double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, 0, res + 1);
        double[] Znodes = GenericBlas.Linspace(-Constants.L, 0, res + 1);
        //double trsh = 0.02 * Constants.ST;
        //double[] Ynodes = GenericBlas.Linspace(0 - trsh - 0.00001, Constants.ST - trsh - 0.00001, res * 2 / 3 + 1);
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST - trsh + 2 * trsh / 9 - 0.00001, Constants.ST + trsh - 2 * trsh / 9 - 0.00001, 7 + 1));
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh - 0.00001, 2 * Constants.L + trsh - 0.00001, res * 4 / 3 + 1));
        //double[] Ynodes = GenericBlas.Linspace(0 - trsh + 0.00001, Constants.ST - trsh + 0.00001, res * 2 / 3 + 1);
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST - trsh + 2 * trsh / 4 + 0.00001, Constants.ST + trsh - 2 * trsh / 4 + 0.00001, 2 + 1));
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh + 0.00001, 2 * Constants.L + trsh + 0.00001, res * 4 / 3 + 1));
        
        double[] Ynodes = GenericBlas.Linspace(0, 2 * Constants.L, 2 * res + 1);

        var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[2] - Xnodes.First()) < 1e-6) || (Math.Abs(X[2]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergencePeriodic(int res, IDatabaseInfo myDb){
        //double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 * res + 1);
        double[] Znodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 * res + 1);
        //double trsh = 0.05 * Constants.ST;
        //double[] Ynodes = GenericBlas.Linspace(0, Constants.ST - 2 * trsh, res + 1);
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh, 2 * Constants.L, 2 * res + 1));
        double[] Ynodes = GenericBlas.Linspace(0, 2 * Constants.L, 2 * res + 1);

        var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes, periodicX: true, periodicZ: true);

        grd.EdgeTagNames.Add(1, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if((Math.Abs(X[1] - 2 * Constants.L) < 1e-8) )
                return 1;
            if((Math.Abs(X[1] - 0.0) < 1e-8) )
                return 2;
            return et;
        });

        //myDb.Controller.DBDriver.SaveGrid(grd, myDb);
        
        return grd;
    }
}

In [21]:
public static class BoundaryAndInitialValueFactory {

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine($"         return 1.0 * ((X[0]-{Constants.X0}).Pow2() + (X[1]-{Constants.Y0}).Pow2() + (X[2]-{Constants.X0}).Pow2() - {Constants.R}.Pow2());");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi1(double[] X, double t){");
            stw.WriteLine($"         return -(X[1] - {Constants.ST});");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }

    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode: GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode: GetPrefixCode());
    }
    static public Formula Get_Phi1() {
        return new Formula("BoundaryAndInitialValues.Phi1", true, AdditionalPrefixCode: GetPrefixCode());
    }
}

## Initialize Controlfiles

First Simulations on a smaller domain, to see if it convegeces towards some particular shape.

In [22]:
//List<ZLS_Control> Controls = new List<ZLS_Control>();

In [23]:
public static HFSI_Control AlandT(int res = 2, int lvl = 2, int p = 2) {
        HFSI_Control C = new HFSI_Control(p);

        // basic database options
        // ======================
        C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
        C.SessionName = "HFSI-Leidenfrost-3D" + "_H" + res + "_AMR" + lvl + "_P" + p + "_Soft_Big";
        C.ProjectName = "HFSI-Leidenfrost";
        C.ProjectDescription = "Leidenfrost on HFSI";
        
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Convtype", "H"));


        // DG degrees
        // ==========
        //C.SetDGdegree(p);

        // Physical Parameters
        // ===================
        C.solveCoupledHeatEquation    = true;
        C.PhysicalParameters.Material = false;

        C.PhysicalParameters.IncludeConvection = true;
        C.ThermalParameters.IncludeConvection  = true;

        C.PhysicalParameters.rho_A = Constants.rho_A;
        C.PhysicalParameters.rho_B = Constants.rho_B;

        C.PhysicalParameters.mu_A = Constants.mu_A;
        C.PhysicalParameters.mu_B = Constants.mu_B;

        C.PhysicalParameters.Sigma = Constants.Sigma;

        C.ThermalParameters.rho_A = Constants.rho_A;
        C.ThermalParameters.rho_B = Constants.rho_B;
        C.ThermalParameters.rho_C = Constants.rho_A;

        C.ThermalParameters.c_A = Constants.c_A;
        C.ThermalParameters.c_B = Constants.c_B;
        C.ThermalParameters.c_C = 0;

        C.ThermalParameters.k_A = Constants.k_A;
        C.ThermalParameters.k_B = Constants.k_B;
        C.ThermalParameters.k_C = Constants.k_A * 1.0e5;

        C.ThermalParameters.hVap  = Constants.hVap;
        C.ThermalParameters.T_sat = Constants.T_sat;

        C.PhysicalParameters.betaS_A = 0.0;
        C.PhysicalParameters.betaS_B = 0.0;

        C.PhysicalParameters.betaL      = 0.0;
        C.PhysicalParameters.sliplength = 0.0;
        C.PhysicalParameters.theta_e    = Math.PI / 2.0; 

        C.Material = new Solid() {
            Density = Constants.rho_A,
            Lame2 = 1e2,
            //Lame2 = 1e11,
            Viscosity = Constants.mu_A
        };

        // grid generation
        // ===============
        bool periodic = false;
        var grd = periodic ? GridFactory.GridConvergencePeriodic(res, BoSSSshell.WorkflowMgm.DefaultDatabase) : GridFactory.GridConvergence(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
        C.SetGrid(grd);
        
        // Initial Values
        // ==============
        C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
        C.AddInitialValue("Phi2", BoundaryAndInitialValueFactory.Get_Phi1());
        C.AddInitialValue("Temperature#A", "X => "+Constants.T_sat, false);
        C.AddInitialValue("Temperature#B", BoundaryAndInitialValueFactory.Get_TempB());
        C.AddInitialValue("Temperature#C", $"X => + {Constants.T_sat} + ({Constants.T_wall}-{Constants.T_sat}) * (1.0- X[1]/{Constants.ST})", false);
        C.AddInitialValue("GravityY#A", new Formula($"X => -{Constants.g}",false));
        C.AddInitialValue("GravityY#B", new Formula($"X => -{Constants.g}",false));
        //C.AddInitialValue("GravityY#C", new Formula($"X => -{Constants.g}",false));


        // boundary conditions
        // ===================
        if(!periodic) C.AddBoundaryValue("freeslip_ZeroGradient");
        C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", "X => " + Constants.T_sat, false);
        if(Constants.ST > 0.0) {
            C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#C", "X => " + Constants.T_wall, false);
        } else {
            C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#B", "X => " + Constants.T_wall, false);
        }


        // level set options
        // =================
        C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
        C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
        C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

        // Timestepping
        // ============
        C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
        C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.BDF2;
        C.dtFixed          = 5.0 * Constants.dt((int)(res * Math.Pow(2, lvl)), p); // Use same timestep for all, better comparability
        //C.dtFixed          = 0.01; 
        C.Endtime          = 1.0;
        C.NoOfTimesteps    = (int)Math.Ceiling(C.Endtime /C.dtFixed);
        //C.NoOfTimesteps    = 5;
        C.saveperiod       = 1;
        C.rollingSaves     = false;
        
        // AMR
        // ============
        int level = lvl;
        C.AdaptiveMeshRefinement = lvl > 0;
        C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 0, maxRefinementLevel = level});
        //C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 1, maxRefinementLevel = level - 1});
        C.AMR_startUpSweeps = lvl;
        //C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRatInnerContactLine { maxRefinementLevel = level, levelSets = new[] { 0, 1 }, FieldWidth = 1 });

        C.DynamicLoadBalancing_On = true;
        C.DynamicLoadBalancing_Period = 20;
        C.DynamicLoadBalancing_RedistributeAtStartup = true;
        C.GridPartType = GridPartType.METIS;

        C.ReInitPeriod = 20;


        // special options
        // ===============
        C.NonLinearSolver.MaxSolverIterations = 20;
        C.NonLinearSolver.MinSolverIterations = 3;
        C.NonLinearSolver.ConvergenceCriterion = 1e-9;
        C.FailOnSolverFail = false;
        //C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MassfluxLogging() { LogPeriod = 1 }); // further logging?

        return C;
}

In [24]:
//DirectoryInfo NativeOverride = null;
//DirectoryInfo ManagedOverride = null;
//var bpc = BoSSSshell.GetDefaultQueue();
//string RelManagedPath = null;

//string mngdir = proj + "_" + DateTime.Now.ToString("MMMdd_HHmmss") + "_managed";
//ManagedOverride = new DirectoryInfo(Path.Combine(bpc.DeploymentBaseDirectory, mngdir));
//ManagedOverride.Create();
//typeof(XNSFE).Assembly.DeployAt(ManagedOverride);
//RelManagedPath = "../" + mngdir + "/" + Path.GetFileName(typeof(XNSFE).Assembly.Location);

//string ntvdir = proj + "_"  + DateTime.Now.ToString("MMMdd_HHmmss") + "_amd64";
//NativeOverride = new DirectoryInfo(Path.Combine(bpc.DeploymentBaseDirectory, ntvdir));
//NativeOverride.Create();
//MetaJobMgrIO.CopyDirectoryRec(ilPSP.Environment.NativeLibraryDir, NativeOverride.FullName, null);

//bpc.DeployRuntime = false; // we did this manually!

In [25]:
//NativeOverride.FullName.Display();
//ManagedOverride.FullName.Display();

In [27]:
var C_CTRLFILE = AlandT(3, 2, 4); // Create control file.
var JobLocal = C_CTRLFILE.CreateJob();

JobLocal.NumberOfMPIProcs = 4;
JobLocal.NumberOfThreads = 4; 

// ===================================
var targetKey = JobLocal.EnvironmentVars.FirstOrDefault(kvp => kvp.Value?.Trim() == "--num_threads").Key;
if (targetKey != null) {
    int valIdx = int.Parse(targetKey.Replace("BOSSS_ARG_", "")) + 1;
    JobLocal.EnvironmentVars[$"BOSSS_ARG_{valIdx}"] = JobLocal.NumberOfThreads.ToString(); 
}
// ===================================

JobLocal.Activate(myBatch);

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job HFSI-Leidenfrost-3D_H3_AMR2_P4_Soft_Big ... 
Deploying executables and additional files ...
Deployment directory: \\fdycluster\miao\databases\deploy\ZLS-Leidenfrost-3D-Paper-Soft-HFSISolver2026Jun30_092816.003118
copied 54 files.
   written file: control.obj
deployment finished.


In [ ]:
//var C_CTRLFILE = AlandT(3,2,2);//Create control file.
//var job = C_CTRLFILE.CreateJob();
//job.NumberOfMPIProcs = 4;
//job.NumberOfThreads = 2;
//job.Activate(myBatch);
//job.ShowOutput();

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1 ... 
Deploying executables and additional files ...
Deployment directory: \\fdygitrunner\BoSSStests\Miao-Temp\deploy\ZLS-Leidenfrost-3D-Paper-Soft-HFSISolver2026Mar25_085245.462719
copied 53 files.
   written file: control.obj
deployment finished.


In [47]:
//BoSSSshell.WorkflowMgm.AllJobs.ForEach(j => j.Value.DeleteOldDeploymentsAndSessions());
//BoSSSshell.WorkflowMgm.AllJobs.Clear();

In [29]:
wmg.DefaultDatabase.Sessions

#0: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart_Restart*	03/26/2026 12:15:50	edf3bc22...
#1: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart*	03/25/2026 15:55:48	eb513a84...
#2: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1*	03/25/2026 08:53:02	befa9b92...
#3: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart_Restart*	03/24/2026 20:17:50	1c7d2dec...
#4: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart_Restart*	03/24/2026 20:15:33	30caa777...
#5: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart*	03/23/2026 01:02:15	e620c881...
#6: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart*	03/23/2026 00:58:43	4293adbc...
#7: ZLS-Leide

In [12]:
wmg.DefaultDatabase.Sessions[8].Timesteps[64]

 { Time-step: 42; Physical time: 0.06851893125290422s; Fields: Phi, PhiDG, Phi2, Phi2DG, VelocityX, VelocityY, VelocityZ, Pressure, Temperature, DisplacementX, DisplacementY, DisplacementZ, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Residual-HeatEq, Residual-DisplaceX, Residual-DisplaceY, Residual-DisplaceZ, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, Curvature, VelocityX@Phi2, VelocityY@Phi2, VelocityZ@Phi2, MPIrank, CutCells; Name:  }

In [8]:
wmg.DefaultDatabase.Sessions[8].Timesteps.Count

641

In [13]:
int ll = 63;
//int kk = 318;

In [14]:
var outPath = wmg.DefaultDatabase.Sessions[8].Timesteps.Every(1).Skip(ll).Export().WithSupersampling(3).Do();
ll = wmg.DefaultDatabase.Sessions[8].Timesteps.Count

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\ZLS-Leidenfrost-3D-Paper-Soft__HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1__f3be2d1c-7782-4cab-92d8-689e1b93a8ab


641

In [174]:
//wmg.DefaultDatabase.Sessions[0].GetDOF(BoSSS.Solution.NSECommon.VariableNames.VelocityX)

In [ ]:
ii//outPath

In [ ]:
7 * wmg.Sessions[1].GetDOF(VariableNames.DisplacementX) + wmg.Sessions[1].GetDOF(BoSSS.Solution.NSECommon.VariableNames.Pressure)

In [ ]:
//BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(86400);

In [ ]:
//var sess = BoSSSshell.WorkflowMgm.Sessions;
//var s = sess.First();
//var exportpath = Path.GetFullPath("./Plots/"+s.Name);
//if(!Directory.Exists(exportpath)) Directory.CreateDirectory(exportpath);
//s.Export().WithShadowFields().WithSupersampling(2).To(exportpath).Do();

## Restart

In [219]:
wmg.DefaultDatabase.Sessions

#0: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart*	3/25/2026 3:55:48 PM	eb513a84...
#1: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1*	3/25/2026 8:53:02 AM	befa9b92...
#2: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart_Restart*	3/24/2026 8:17:50 PM	1c7d2dec...
#3: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart_Restart*	3/24/2026 8:15:33 PM	30caa777...
#4: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart*	3/23/2026 1:02:15 AM	e620c881...
#5: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart*	3/23/2026 12:58:43 AM	4293adbc...
#6: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1	3/21/2026 10:07:40 PM	eb6b9fbc...
#7: ZLS-Leidenfrost-3D-Pape

In [220]:
var Sloc = wmg.DefaultDatabase.Sessions[0];
Sloc

ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart*	3/25/2026 3:55:48 PM	eb513a84...

In [221]:
var c8 = Sloc.CreateRestartControl() as HFSI_Control;

In [222]:
c8.GetType()

HFSISolver.HFSI_Control

In [223]:
c8.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 0, maxRefinementLevel = 4});
c8.AMR_startUpSweeps = 4;


In [224]:
c8.Endtime          = 3.0;
c8.NoOfTimesteps    = (int)Math.Ceiling(c8.Endtime /c8.dtFixed);

In [225]:
c8.GridGuid

6305bba4-7562-43a1-b667-30b310809c7d

In [226]:
Sloc.Timesteps.Last().GridID

9df3b979-3cd8-4e1f-9c36-5baf232e5562

In [227]:
c8.GridGuid = Sloc.Timesteps.Last().GridID;

In [228]:
c8.GridGuid

9df3b979-3cd8-4e1f-9c36-5baf232e5562

In [229]:
var JobLocal8 = c8.CreateJob();
JobLocal8.Status

PreActivation

In [230]:
JobLocal8.NumberOfMPIProcs = 8;
JobLocal8.NumberOfThreads = 2;
JobLocal8.Activate(myBatch);
//JobLocal2.ShowOutput();

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart_Restart ... 
Deploying executables and additional files ...
Deployment directory: \\fdygitrunner\BoSSStests\Miao-Temp\deploy\ZLS-Leidenfrost-3D-Paper-Soft-HFSISolver2026Mar26_121529.028148
copied 53 files.
   written file: control.obj
deployment finished.


In [260]:
wmg.DefaultDatabase.Sessions

#0: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart_Restart*	3/26/2026 12:15:50 PM	edf3bc22...
#1: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart*	3/25/2026 3:55:48 PM	eb513a84...
#2: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1*	3/25/2026 8:53:02 AM	befa9b92...
#3: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart_Restart*	3/24/2026 8:17:50 PM	1c7d2dec...
#4: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart_Restart*	3/24/2026 8:15:33 PM	30caa777...
#5: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart*	3/23/2026 1:02:15 AM	e620c881...
#6: ZLS-Leidenfrost-3D-Paper-Soft	HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart*	3/23/2026 12:58:43 AM	4293adbc...
#7: 

In [ ]:
//int iii = 0;
//int jjj = 0;
//int kkk = 0;

In [273]:
var outPath = wmg.DefaultDatabase.Sessions[0].Timesteps.Every(1).Skip(kkk).Export().WithSupersampling(3).Do();
kkk = wmg.DefaultDatabase.Sessions[0].Timesteps.Count

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\ZLS-Leidenfrost-3D-Paper-Soft__HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Small_MK1_Restart_Restart__edf3bc22-73b5-42b7-b880-fd2d4dd52efb


591

In [270]:
var outPath = wmg.DefaultDatabase.Sessions[3].Timesteps.Every(1).Skip(iii).Export().WithSupersampling(3).Do();
iii = wmg.DefaultDatabase.Sessions[3].Timesteps.Count

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\ZLS-Leidenfrost-3D-Paper-Soft__HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_Big_MK1_Restart_Restart__1c7d2dec-aaec-4d8e-a53a-89874bf544b6


115

In [271]:
var outPath = wmg.DefaultDatabase.Sessions[4].Timesteps.Every(1).Skip(jjj).Export().WithSupersampling(3).Do();
jjj = wmg.DefaultDatabase.Sessions[4].Timesteps.Count

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\ZLS-Leidenfrost-3D-Paper-Soft__HFSI-Leidenfrost-3D_H3_AMR2_P2_Soft_test_ReInit_20_Fast_MK1_Restart_Restart__30caa777-2244-4199-9b06-c04eb8a3c05e


239